## **Replication Project - Alex Eyre**
### Tumor immune evasion arises through loss of TNF sensitivity
### (Kearney et al., Science Immunology 2018)
=============================================================================

In [1]:
# =============================================================================
# Import libraries.
# =============================================================================
#import os
#import sys
#import logging
import time
import warnings
import pandas as pd
from pathlib import Path

# Gene Annotation Utilities
import mygene
import gseapy as gp
from gprofiler import GProfiler

# Raw Read, RNA-seq, CRISPR screen comparison analyses.
# The authors do not provide raw CRISPR screen RNA-seq data.
#import mageck
#sys.path.insert(0, "/mnt/c/Users/aweyr/OneDrive/Documents/Programs/DrugZ")
#import drugz as dz

In [2]:
# =============================================================================
# Startup R.
# =============================================================================
#logging.getLogger("rpy2").setLevel(logging.CRITICAL)
#%load_ext rpy2.ipython

In [3]:
#%%R
# =============================================================================
# Load R Packages.
# =============================================================================
#suppressPackageStartupMessages({library("DESeq2") # DESeq2: 1.50.2
#                               library("clusterProfiler") # clusterProfiler: 4.18.4
#                               library("ComplexHeatmap") # ComplexHeatmap: 2.26.1
#                               library("enrichplot") # enrichplot: 1.30.5 
#                               library("org.Mm.eg.db") # org.Mm.eg.db: 1.30.5 
#                               library("CellChat")}) # CellChat

=============================================================================
#### Functions
=============================================================================

In [4]:
def excel_folder_to_csv(
    input_dir,
    output_dir=None,
    sheet=0,
    recursive=False,
    overwrite=False,
    drop_empty=True,
):
    
    """
    Convert Excel workbooks in a directory to CSV files.

    Parameters
    ----------
    input_dir : str or Path
        Directory containing Excel files.
        
    output_dir : str or Path, optional
        Directory to save CSV files. If None, CSVs are saved alongside
        the Excel files.
        
    sheet : int or str, default=0
        Worksheet index or name to convert.
        
    recursive : bool, default=False
        Search subdirectories for Excel files.
        
    overwrite : bool, default=False
        Overwrite existing CSV files.
        
    drop_empty : bool, default=True
        Remove completely empty rows and columns before saving.

    Returns
    -------
    list[Path]
        Paths to the CSV files created.
    """

    input_dir = Path(input_dir)

    if output_dir is None:
        output_dir = input_dir
    else:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

    pattern = "**/*.xlsx" if recursive else "*.xlsx"

    csv_files = []

    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="Unknown extension is not supported and will be removed",
            category=UserWarning,
        )

        for excel_file in input_dir.glob(pattern):

            csv_file = output_dir / f"{excel_file.stem}.csv"

            if csv_file.exists() and not overwrite:
                print(f"Skipping: {csv_file.name}")
                continue

            print(f"Converting: {excel_file.name}")

            df = pd.read_excel(
                excel_file,
                sheet_name=sheet,
            )

            if drop_empty:
                df = (
                    df
                    .dropna(axis=0, how="all")  # Remove empty rows
                    .dropna(axis=1, how="all")  # Remove empty columns
                )

            df.to_csv(csv_file, index=False)

            csv_files.append(csv_file)

    print(f"\nCreated {len(csv_files)} CSV file(s).")

    return csv_files

In [5]:
def parse_mageck_results(
    df: pd.DataFrame,
    *,
    header: bool = True,
    columns_per_comparison: int = 14,
    reset_index: bool = True,
) -> dict[str, pd.DataFrame]:
    
    """
    Parse one or more MAGeCK comparison results from a results table.

    Parameters
    ----------
    df : pd.DataFrame
        Raw MAGeCK results table read with header=None.

    header : bool
        Whether the table contains an additional first row with comparison
        names.

        If True, row 0 contains comparison names and row 1 contains the
        MAGeCK output headers.

        If False, row 0 contains the MAGeCK output headers.

    columns_per_comparison : int
        Number of columns in each MAGeCK comparison block.

    reset_index : bool
        Whether to reset the index after removing metadata/header rows.

    Returns
    -------
    dict[str, pd.DataFrame]
        Dictionary where keys are comparison names and values are cleaned
        MAGeCK result tables.
    """

    mageck_results = {}

    n_comparisons = df.shape[1] // columns_per_comparison

    for comparison_idx in range(n_comparisons):
        start_idx = (comparison_idx * columns_per_comparison) + comparison_idx
        end_idx = start_idx + columns_per_comparison

        subset = df.iloc[:, list(range(start_idx, end_idx))].copy()
        
        if header:
            comparison_name = str(subset.columns[0]).strip() 
            subset.columns = subset.iloc[0]
            subset = subset.iloc[1:]
        else:
            comparison_name = f"comparison_{comparison_idx + 1}"

        # Clean column names
        subset.columns = subset.columns.astype(str).str.strip()

        # Fix possible malformed id column
        if subset.columns[0] != "id":
            subset = subset.rename(columns={subset.columns[0]: "id"})

        # Convert MAGeCK statistic columns to numeric
        for col in subset.columns:
            if col != "id":
                subset[col] = pd.to_numeric(subset[col], errors="coerce")

        if reset_index:
            subset = subset.reset_index(drop = True)

        mageck_results[comparison_name] = subset

    return mageck_results

In [6]:
def curate_gene_list(
    df,
    gene_column = "id",
    sort_by = None,
    ascending = True,
    top_n = None,
):

    """
    Create a curated list of unique gene symbols from a DataFrame.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing gene information.

    gene_column : str, default="id"
        Column containing gene symbols.

    sort_by : str, optional
        Column used to sort the DataFrame before extracting genes.
        If None, the original row order is preserved.

    ascending : bool, default=True
        Sort order used when sort_by is specified.

    top_n : int, optional
        Number of genes to retain after sorting. If None, all genes
        are retained.

    Returns
    -------
    list
        List of unique gene symbols in the specified order.
    """

    # Create a copy of the DataFrame to avoid modifying the original.
    gene_df = df.copy()

    # Sort the DataFrame if a sorting column is provided.
    if sort_by is not None:

        gene_df = gene_df.sort_values(
            sort_by,
            ascending = ascending,
        )

    # Keep only the top N genes if requested.
    if top_n is not None:

        gene_df = gene_df.head(top_n)

    # Extract the gene symbols, remove missing values and duplicates,
    # and convert the result to a list.
    gene_list = (
        gene_df[gene_column]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .tolist()
    )

    return gene_list

In [7]:
def get_organism(
    organism,
    package,
):

    """
    This standardization step suggested and provided by ChatGPT.
    """
    
    organism_map = {
        "human": {
            "mygene": "human",
            "gprofiler": "hsapiens",
            "enrichr": "human",
        },
        "mouse": {
            "mygene": "mouse",
            "gprofiler": "mmusculus",
            "enrichr": "mouse",
        },
    }

    return organism_map[organism.lower()][package.lower()]

In [8]:
def convert_mouse_to_human_homologs(
    gene_list,
):
    
    """
    Convert mouse gene symbols to predicted human orthologs using g:Profiler.

    Parameters
    ----------
    gene_list : list-like
        Mouse gene symbols.

    Returns
    -------
    pandas.DataFrame
        Table linking input mouse genes to human ortholog gene symbols.
    """

    gp_client = GProfiler(
        return_dataframe = True
    )

    homologs = gp_client.orth(
        organism = "mmusculus",
        query = gene_list,
        target = "hsapiens",
    )

    homologs = homologs.rename(
        columns = {
            "input": "mouse_gene",
            "converted": "human_gene",
            "name": "human_gene_name",
            "description": "human_gene_description",
            "ortholog_ensg": "human_ensembl_gene_id",
        }
    )

    return homologs

In [9]:
def genesymbol_to_geneid(
    gene_list,
    organism = "hsapiens",
    target_namespace = "ENSG",
):
    
    """
    Convert gene identifiers using g:Profiler.

    Parameters
    ----------
    gene_list : list-like
        List of gene identifiers (e.g., gene symbols).

    organism : str, default="hsapiens"
        Organism name recognized by g:Profiler.

    target_namespace : str, default="ENSG"
        Identifier type to convert to.

        Common options:
            ENSG                Ensembl Gene ID
            ENST                Ensembl Transcript ID
            ENTREZGENE          NCBI Entrez Gene ID
            UNIPROTSWISSPROT    Reviewed UniProt accession

    Returns
    -------
    pandas.DataFrame
        Table of converted identifiers.
    """

    gp_client = GProfiler(return_dataframe=True)

    ids = gp_client.convert(
        organism = organism,
        query = gene_list,
        target_namespace = target_namespace,
    )

    ids = ids.rename(columns={
        "incoming": "input_gene",
        "converted": "converted_id",
        "n_incoming": "num_input_matches",
        "n_converted": "num_output_matches",
        "name": "converted_name",
        "description": "converted_description",
        "namespace": "target_namespace",
    })

    return ids

In [10]:
def get_enrichr_geneset_libraries(
    organism = "Human",
    search_terms = None,
):
    
    """
    Retrieve available Enrichr gene set libraries and optionally
    filter them by one or more search terms.

    Parameters
    ----------
    organism : str, default="Human"
        Organism supported by gseapy.

    search_terms : str or list of str, optional
        Keyword(s) used to filter the available libraries.

        Examples:
            "GO"
            ["GO", "KEGG", "Reactome"]

        If None, all libraries are returned.

    Returns
    -------
    list
        List of matching gene set library names.
    """

    libraries = gp.get_library_name(organism=organism)

    if search_terms is None:
        return sorted(libraries)

    # Allow a single string or a list of strings
    if isinstance(search_terms, str):
        search_terms = [search_terms]

    filtered = [
        library
        for library in libraries
        if any(term.lower() in library.lower() for term in search_terms)
    ]

    return sorted(filtered)

In [11]:
def annotate_pathways_enrichr(
    gene_list,
    gene_sets,
    organism = "human",
    top_n = None,
    outdir = None,
    sleep_seconds = 3
):
    
    """
    Run Enrichr pathway/gene-set enrichment using gseapy.

    Parameters
    ----------
    gene_list : list-like
        List of gene symbols to test for enrichment.

    gene_sets : str or list of str, optional
        Enrichr gene set library or libraries.

    organism : str, default = "Human"
        Organism supported by Enrichr.

    top_n : int, optional
        If provided, only the first top_n genes are used.

    outdir : str or None, default = None
        Output directory. If None, results are not written to disk.

    sleep_seconds: int, default = 3
        Time between database retrieval requests.

    Returns
    -------
    pandas.DataFrame
        Enrichr enrichment results.
    """
    
    genes = (
        pd.Series(gene_list)
        .dropna()
        .astype(str)
        .drop_duplicates()
        .tolist()
    )

    if top_n is not None:
        genes = genes[:top_n]

    if isinstance(gene_sets, str):
        gene_sets = [gene_sets]

    results = []

    for gene_set in gene_sets:

        try:

            enrichr_results = gp.enrichr(
                gene_list = genes,
                gene_sets = [gene_set],
                organism = organism,
                outdir = outdir,
            )

            result = enrichr_results.results.copy()
            result["gene_set_library"] = gene_set

            results.append(result)

            time.sleep(sleep_seconds)

        except Exception as e:

            print(f"Failed: {gene_set} — {e}")
            time.sleep(sleep_seconds)

    if len(results) == 0:
        return pd.DataFrame()

    return pd.concat(
        results,
        ignore_index = True,
    )

In [12]:
def annotate_genes_mygene(
    gene_list, 
    species = "human"
):
    
    """
    Annotate a list of gene symbols using MyGene.info.

    Parameters
    ----------
    gene_list : list-like
        List of gene symbols.

    species : str, default = "human"
        Species name accepted by MyGene.info.
        Additional: "mouse"

    Returns
    -------
    pandas.DataFrame
        Gene annotation table.
    """

    mg = mygene.MyGeneInfo()

    annot = mg.querymany(
        gene_list,
        scopes="symbol",
        species=species,
        fields=[
            "symbol",
            "name",
            "entrezgene",
            "ensembl.gene",
            "alias",
            "summary",
            "uniprot.Swiss-Prot",
            "go.BP.term",
            "go.MF.term",
            "go.CC.term",
            "pathway.reactome",
            "pathway.kegg",
        ],
        as_dataframe=True,
        df_index=False,
    )

    annot = annot.rename(columns={
        "query": "input_gene",
        "symbol": "official_symbol",
        "name": "gene_name",
        "entrezgene": "entrez_id",
        "ensembl.gene": "ensembl_id",
        "uniprot.Swiss-Prot": "uniprot_swissprot",
        "go.BP.term": "go_bp",
        "go.MF.term": "go_mf",
        "go.CC.term": "go_cc",
        "pathway.reactome": "reactome_pathways",
        "pathway.kegg": "kegg_pathways",
    })

    annot = annot.drop_duplicates(subset="input_gene")

    return annot

In [13]:
def annotate_genes_enrichr(
    enrichr_results,
    genes_column = "Genes",
):
    
    enrichr_long = enrichr_results.copy()

    enrichr_long[genes_column] = (
        enrichr_long[genes_column]
        .astype(str)
        .str.split(";")
    )

    enrichr_long = enrichr_long.explode(
        genes_column
    ).reset_index(drop = True)

    enrichr_long = enrichr_long.rename(
        columns = {
            genes_column: "input_gene"
        }
    )

    return enrichr_long

In [14]:
def annotate_genes_HGNC(
    gene_list,
    gene_annotation,
    symbol_column = "Approved symbol",
):
    
    """
    Annotate a list of gene symbols using an HGNC gene annotation table.

    Parameters
    ----------
    gene_list : list-like
        List of gene symbols to annotate.

    gene_annotation : pandas.DataFrame
        HGNC gene annotation table.

    symbol_column : str, default="Approved symbol"
        Column in the HGNC annotation table containing the official
        HGNC-approved gene symbols.

    Returns
    -------
    pandas.DataFrame
        HGNC annotation table containing one row for each gene in
        gene_list, preserving the original gene order.
    """

    # Convert gene list into a DataFrame and preserve original order.
    genes = pd.DataFrame(
        {"input_gene": gene_list}
    )

    genes["input_gene_order"] = range(len(genes))

    # Create uppercase matching keys.
    genes["gene_match_key"] = (
        genes["input_gene"]
        .astype(str)
        .str.upper()
    )

    gene_annotation = gene_annotation.copy()

    gene_annotation["gene_match_key"] = (
        gene_annotation[symbol_column]
        .astype(str)
        .str.upper()
    )

    # Merge using the normalized matching key.
    annotation_subset = genes.merge(
        gene_annotation,
        on = "gene_match_key",
        how = "left",
    )

    # Sort back to original gene-list order.
    annotation_subset = annotation_subset.sort_values(
        "input_gene_order"
    ).reset_index(drop = True)

    # Remove helper columns.
    annotation_subset = annotation_subset.drop(
        columns = [
            "input_gene_order",
            "gene_match_key",
        ]
    )

    return annotation_subset

In [15]:
def annotate_genes_ligandreceptor(
    gene_list,
    ligand_receptor,
    source_column = "source (gene ID)",
    target_column = "target (gene ID)",
):
    
    """
    Annotate a list of gene symbols using a ligand-receptor interaction
    database.

    Parameters
    ----------
    gene_list : list-like
        List of gene symbols to annotate.

    ligand_receptor : pandas.DataFrame
        Ligand-receptor interaction database.

    source_column : str, default="source"
        Column containing ligand/source gene symbols.

    target_column : str, default="target"
        Column containing receptor/target gene symbols.

    Returns
    -------
    pandas.DataFrame
        Ligand-receptor annotation table containing one or more rows for
        each gene in gene_list. Genes may appear multiple times if they
        participate in multiple ligand-receptor interactions.
    """

    # Convert the gene list into a DataFrame and preserve original order.
    genes = pd.DataFrame(
        {"input_gene": gene_list}
    )

    genes["input_gene_order"] = range(len(genes))

    # Create uppercase matching keys for input genes.
    genes["gene_match_key"] = (
        genes["input_gene"]
        .astype(str)
        .str.upper()
    )

    ligand_receptor = ligand_receptor.copy()

    # Create source and target matching keys.
    ligand_receptor["source_match_key"] = (
        ligand_receptor[source_column]
        .astype(str)
        .str.upper()
    )

    ligand_receptor["target_match_key"] = (
        ligand_receptor[target_column]
        .astype(str)
        .str.upper()
    )

    # Match input genes to source/ligand genes.
    source_hits = genes.merge(
        ligand_receptor,
        left_on = "gene_match_key",
        right_on = "source_match_key",
        how = "left",
    )

    source_hits["lr_role"] = "source"

    # Match input genes to target/receptor genes.
    target_hits = genes.merge(
        ligand_receptor,
        left_on = "gene_match_key",
        right_on = "target_match_key",
        how = "left",
    )

    target_hits["lr_role"] = "target"

    # Combine source and target annotations.
    ligand_receptor_subset = pd.concat(
        [source_hits, target_hits],
        ignore_index = True,
    )

    # Remove rows where no ligand-receptor match was found.
    ligand_receptor_subset = ligand_receptor_subset[
        ligand_receptor_subset[source_column].notna()
        | ligand_receptor_subset[target_column].notna()
    ].copy()

    # Restore original gene order.
    ligand_receptor_subset = ligand_receptor_subset.sort_values(
        "input_gene_order"
    ).reset_index(drop = True)

    # Remove helper columns.
    ligand_receptor_subset = ligand_receptor_subset.drop(
        columns = [
            "input_gene_order",
            "gene_match_key",
            "source_match_key",
            "target_match_key",
        ],
        errors = "ignore",
    )

    return ligand_receptor_subset

=============================================================================
#### Immune evasion occurs through loss of TNF, IFN-y, or antigen presentation pathways
=============================================================================

In [16]:
# =============================================================================
# Define project paths.
# =============================================================================
project_dir = Path.cwd().parents[0]
processed_dir = project_dir / "data"/ "processed"
supp_dir = project_dir / "data" / "supplemental"
annotation_dir = project_dir / "data" / "annotation"

In [17]:
# =============================================================================
# Parse in gene annotation data.
# =============================================================================
# HUGO Gene Nomenclature Committee (HGNC) Gene Annotation Dataset
gene_annotation_data = pd.read_csv(annotation_dir / "gene annotation.txt", sep = "\t")

# Ligand-Receptor Database
ligand_receptor_data = pd.read_csv(annotation_dir / "ligand_receptor.csv", low_memory = False)
ligrec_gene_list = pd.read_csv(annotation_dir / "genes.csv")

In [18]:
# =============================================================================
# Parse in processed and normalized RNA-seq Files (https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3062996)
# =============================================================================

# These do not have anything to do with the CRISPR screening.   :(

# Investigation of the transcriptional response of wild type or Ado knockout MC38 cells to TNF.
WTvsTNF_sampleinfo = pd.read_csv(processed_dir / "GSE112251_samples.txt", sep = "\t")
WTvsTNF_norm_data = pd.read_csv(processed_dir / "GSE112251_171123_NB501056_0081_AHWWTVBGX3_normalizedcountdata.csv")

# Investigation of the transcriptional response to TNF and Interferon gamma or T cells.
MC38_sampleinfo = pd.read_csv(processed_dir / "GSE112252_samples.txt", sep = "\t")   
MC38_coculture_norm_data = pd.read_csv(processed_dir / "GSE112252_170605_NB501056_0018_AH2FYTBGX3_MC38_coculture_normalizedcounts.csv")
MC38_invitro_norm_data = pd.read_csv(processed_dir / "GSE112252_170605_NB501056_0018_AH2FYTBGX3_MC38_invitro_normalizedcounts.csv")

In [19]:
# =============================================================================
# Parse in published supplemental tables (MAGeCK Results).   CHECK THESE
# =============================================================================
mageck_B16perfKO_OT1 = parse_mageck_results(pd.read_csv(supp_dir / "kearney18_s2_crispr_B16_perfKO_OT1.csv"), header = False, columns_per_comparison = 11)
mageck_B16_OT1 = parse_mageck_results(pd.read_csv(supp_dir / "kearney18_s3_crispr_B16_OT1.csv"), header = False, columns_per_comparison = 11)
mageck_MC38_OT1 = parse_mageck_results(pd.read_csv(supp_dir / "kearney18_s4_crispr_MC38_OT1.csv"), header = False, columns_per_comparison = 11)
mageck_MDA_CART = parse_mageck_results(pd.read_csv(supp_dir / "kearney18_s5_crispr_MDA_CART.csv"), header = False, columns_per_comparison = 11)
mageck_OT1_IgGvT0 = parse_mageck_results(pd.read_csv(supp_dir / "kearney18_s6_crispr_OTI_IgGvT0.csv"), header = False, columns_per_comparison = 11)
mageck_OTI_PD1vT0 = parse_mageck_results(pd.read_csv(supp_dir / "kearney18_s7_crispr_OTI_PD1vT0.csv"), header = False, columns_per_comparison = 11)
mageck_OTI_PD1vT0 = parse_mageck_results(pd.read_csv(supp_dir / "kearney18_s8_crispr_MC38_NK.csv"), header = False, columns_per_comparison = 11)

#### MAGeCK Output Description
```
id: gene symbol analyzed by MAGeCK.
num: number of sgRNAs assigned to that gene.
neg|score: RRA score for negative selection/depletion.
neg|p-value: p-value for negative selection.
neg|fdr: adjusted p-value for negative selection.
neg|rank: rank among negatively selected genes.
neg|goodsgrna: sgRNAs supporting negative selection.
pos|score: RRA score for positive selection/enrichment.
pos|p-value: p-value for positive selection.
pos|fdr: adjusted p-value for positive selection.
pos|rank: rank among positively selected genes.
pos|goodsgrna: sgRNAs supporting positive selection.
```

In [20]:
# =============================================================================
# Select Gene List.
# =============================================================================
gene_list = curate_gene_list(
    mageck_MC38_OT1["comparison_1"],
    gene_column = "id",
    sort_by = "pos|rank",
    ascending = True,
    top_n = 50,
)

In [21]:
# =============================================================================
# Identify human homologs of mouse gene set.
# =============================================================================
gene_list_human = convert_mouse_to_human_homologs(
    gene_list
)

In [22]:
# =============================================================================
# Convert Gene Identifiers.
# =============================================================================
# Convert Gene Symbols to Ensemble IDs
gprofiler_ids = genesymbol_to_geneid(gene_list,
                                     organism = "hsapiens",
                                     target_namespace = "ENSG")

#### gprofiler_ids Output Description (DataFrame)
```
input_gene: original gene symbol submitted.
converted_id: converted identifier, usually Ensembl gene ID.
num_input_matches: number of input identifiers matching this query.
num_output_matches: number of converted IDs returned.
converted_name: gene name from g:Profiler.
converted_description: longer gene description.
target_namespace: identifier namespace returned.
query: original query index/order information.
```

In [23]:
# =============================================================================
# Get gseapy pathway libraries.
# =============================================================================
# Retrieve a list of mouse-related gene libraries from gseapy
# Whose library names contain: Reactome, Kegg, and GO_Biological
gp_libraries = get_enrichr_geneset_libraries(organism = "mouse",
                                             search_terms = [
                                                 "GO_Biological",
                                                 "Reactome",
                                                 "KEGG"
                                             ]
                                            )

In [24]:
# =============================================================================
# Perform enrichr pathway enrichment annotation for each pathway library.
# =============================================================================
annotation_pathway_enrichr = annotate_pathways_enrichr(
    gene_list,
    gene_sets = gp_libraries,
    organism = "mouse",
    sleep_seconds = 1
)

#### annotation_pathway_enrichr Output Description (DataFrame)
```
Gene_set: Name of the Enrichr gene set library that produced the enrichment result.
Term: Name of the enriched biological pathway, Gene Ontology term, or gene set.
Overlap: Number of input genes overlapping the term divided by the total number of genes
    annotated to that term (e.g., "7/74").
P-value: Nominal enrichment p-value calculated using Fisher's exact test.
Adjusted P-value: Multiple-testing corrected p-value (Benjamini-Hochberg False Discovery
    Rate).
Old P-value: Legacy p-value reported by the Enrichr API for compatibility with previous
    versions. Typically identical to or superseded by the current P-value.
Old Adjusted P-value: Legacy multiple-testing corrected p-value retained for compatibility.
    Typically identical to or superseded by the current Adjusted P-value.
Odds Ratio: Measure of the strength of enrichment. Larger values indicate a stronger
    association between the input gene list and the enriched term.
Combined Score: Enrichr ranking metric calculated from the p-value and deviation from the
    expected rank. Higher scores indicate stronger overall enrichment.
Genes: Semicolon-separated list of input genes contributing to the enrichment of
    the corresponding term.
gene_set_library: Gene set library queried by the annotation function. This column is added
    by the wrapper function to identify the source library when results from multiple 
    libraries are combined.
```

In [25]:
# =============================================================================
# Perform enrichr gene enrichment annotation for each library.
# =============================================================================
annotation_gene_enrichr = annotate_genes_enrichr(
    annotation_pathway_enrichr
)

#### annotation_gene_enrichr Output Description (DataFrame)
```
Gene_set: Name of the Enrichr gene set library that produced the enrichment result.
Term: Enriched pathway, Gene Ontology term, or gene set associated with this gene.
Overlap: Number of submitted genes overlapping the term divided by the total number
    of genes annotated to that term.
P-value: Nominal enrichment p-value for the full enriched term.
Adjusted P-value: Multiple-testing corrected p-value for the full enriched term.
Old P-value: Legacy p-value reported by the Enrichr API for compatibility.
Old Adjusted P-value: Legacy adjusted p-value reported by the Enrichr API for compatibility.
Odds Ratio: Strength of enrichment for the full enriched term.
Combined Score: Enrichr combined ranking score for the full enriched term.
input_gene: Individual gene from the submitted gene list that overlaps the enriched term.
gene_set_library: Gene set library queried by your wrapper function.
```

In [26]:
# =============================================================================
# Perform mygene Annotation.
# =============================================================================
annotation_mygene = annotate_genes_mygene(
    gene_list,
    species = "mouse"
)

2 input query terms found no hit:	['AU022751', 'Fam26f']


#### mygene Output Description (DataFrame)
```
input_gene: Original gene symbol submitted to MyGene.info.
_id: MyGene.info internal gene record identifier, often the Entrez Gene ID.
_score: MyGene.info match score for the query result.
alias: Alternative gene symbols or aliases associated with the gene.
entrez_id: NCBI Entrez Gene identifier.
gene_name: Full descriptive gene name.
summary: Short functional summary of the gene.
official_symbol: Official gene symbol returned by MyGene.info.
ensembl_id: Ensembl gene identifier extracted from `ensembl.gene`.
go.BP: Gene Ontology Biological Process annotations.
go.CC: Gene Ontology Cellular Component annotations.
go.MF: Gene Ontology Molecular Function annotations.
pathway.kegg.id: KEGG pathway identifier.
pathway.kegg.name: KEGG pathway name.
uniprot_swissprot: Reviewed UniProt Swiss-Prot accession.
reactome_pathways: Reactome pathway annotations after renaming `pathway.reactome`.
notfound: Indicates whether MyGene.info failed to find a match for the input gene.
go_mf: Renamed version of `go.MF.term`; Molecular Function GO terms.
kegg_pathways: Renamed version of `pathway.kegg`; KEGG pathway annotations.
go_cc: Renamed version of `go.CC.term`; Cellular Component GO terms.
ensembl: Full Ensembl annotation object returned by MyGene.info.
go: Full Gene Ontology annotation object returned by MyGene.info.
uniprot.Swiss-Prot: Raw reviewed UniProt Swiss-Prot accession field before renaming.
go.BP.term: Biological Process GO term names.
go.MF.term: Molecular Function GO term names.
go.CC.term: Cellular Component GO term names.
pathway.reactome: Raw Reactome pathway annotations returned by MyGene.info.
pathway.kegg: Raw KEGG pathway annotations returned by MyGene.info.
```

#### enrichr Output Description (DataFrame)
```
Gene_set: enrichment library queried.
Term: enriched pathway, process, or gene set.
Overlap: fraction of submitted genes overlapping the term.
P-value: nominal enrichment p-value.
Adjusted P-value: multiple-testing-adjusted p-value.
Odds Ratio: strength of enrichment.
Combined Score: Enrichr combined ranking score.
Genes: submitted genes overlapping the term.
Gene_set_library: library queried by the wrapper function.
```

In [27]:
# =============================================================================
# Build "gene_annotation.txt" library.
# =============================================================================
annotation_HGNC = annotate_genes_HGNC(
    gene_list,
    gene_annotation_data
)

#### "gene_annotation.txt" Description (DataFrame)
```
HGNC: HGNC gene identifier assigned by the HUGO Gene Nomenclature Committee.
ID: Numeric or database-specific identifier associated with the HGNC record.
Status: Current approval status of the gene symbol.
Approved symbol: Official HGNC-approved gene symbol.
Approved name: Full HGNC-approved gene name.
Alias symbol: Alternative gene symbols or commonly used synonyms.
Alias name: Alternative full gene names or descriptive aliases.
Previous symbol: Former gene symbols that have been replaced by the approved symbol.
Previous name: Former gene names that have been replaced by the approved name.
Chromosome: Chromosome on which the gene is located.
Chromosome location: Cytogenetic location or chromosomal band.
Locus group: Broad gene category, such as protein-coding gene, non-coding RNA, pseudogene, or phenotype.
HGNC family name: Gene family or grouping assigned by HGNC.
HGNC family ID: HGNC identifier for the associated gene family.
UniProt accession: UniProt protein accession associated with the gene.
Ensembl gene ID: Ensembl stable gene identifier.
RefSeq accession: NCBI RefSeq transcript or gene accession associated with the gene.
```

In [28]:
# =============================================================================
# Build "ligand-receptor.csv" Ligand library.
# =============================================================================
annotation_ligrec_ligands = annotate_genes_ligandreceptor(
    gene_list,
    ligand_receptor_data,
    source_column = "source (gene ID)",
    target_column = "target (gene ID)"
)

In [30]:
# =============================================================================
# Build "ligand-receptor.csv" Receptor library.
# =============================================================================
annotation_ligrec_receptors = annotate_genes_ligandreceptor(
    gene_list,
    ligand_receptor_data,
    source_column = "target (gene ID)",
    target_column = "source (gene ID)"
)

#### "ligand_receptor.csv" Description (DataFrame)
```
source (gene ID): Source gene in the interaction, typically encoding the ligand or signaling molecule that initiates communication.
target (gene ID): Target gene in the interaction, typically encoding the receptor or downstream binding partner.
LR_pair: Ligand-receptor pair identifier, usually combining the source and target gene symbols.
is_stimulation: Indicates whether the interaction is stimulatory or activating (Boolean).
is_inhibition: Indicates whether the interaction is inhibitory or suppressive (Boolean).
interaction: Description or classification of the molecular interaction between the source and target genes.
evidence: Experimental or literature evidence supporting the interaction.
data source: Original database or publication from which the ligand-receptor interaction was obtained.
```